# 05 · Readability  _(owner: M5 — Laura)_

**Goal:** understand how our readability metrics behave before tuning the Readability agent (target: Flesch-Kincaid ≤ 8 on ≥ 80% of outputs, without changing facts).

1. What `readability_scores()` and `meets_readability_target()` measure
2. Jargon-heavy vs. plain version of the same sentence
3. 20 PLABA pairs (technical abstract vs. human plain rewrite) — score both sides, chart grade levels

In [ ]:
# One-time: newer textstat needs the CMU pronouncing dictionary for syllable counts
import nltk
nltk.download("cmudict", quiet=True)

import pandas as pd
import matplotlib.pyplot as plt
from clearmed.eval_harness import readability_scores, meets_readability_target

## 1 · What the harness measures

| Metric | What it is | Formula idea |
|---|---|---|
| **Flesch-Kincaid grade** | US school grade needed to read it | `0.39 × words/sentence + 11.8 × syllables/word − 15.59` |
| **SMOG** | Grade level from polysyllabic (3+ syllable) words | common in health-literacy research |
| **Jargon density** | share of words *not* on the Dale-Chall list of ~3,000 familiar words | proxy for jargon, not medical-specific |
| **Word count** | length | |

`meets_readability_target(text)` → `True` if FK grade ≤ 8.

⚠️ Note: these are **surface formulas** — they only count sentence length and syllables. They don't know if the text is actually understandable, correct, or complete. (Like Lexile scores for kids' books.)

## 2 · Same sentence, two versions

In [ ]:
jargon = ("Patient was prescribed a beta-adrenergic antagonist for management of essential "
          "hypertension; follow-up with cardiology is recommended in two weeks.")
plain = "You got a new medicine to lower your blood pressure. See your heart doctor in two weeks."

compare = pd.DataFrame({"jargon": readability_scores(jargon), "plain": readability_scores(plain)})
compare.loc["meets_target"] = [meets_readability_target(jargon), meets_readability_target(plain)]
compare

## 3 · 20 PLABA pairs (technical abstract → human plain rewrite)

Source: PLABA test split from OSF (https://osf.io/rnpmf), saved as `data/plaba_test.csv`. One adaptation per abstract, first 20 abstracts.

In [ ]:
plaba = pd.read_csv("../data/plaba_test.csv").drop_duplicates("pmid").head(20).reset_index(drop=True)

rows = []
for i, r in plaba.iterrows():
    t, p = readability_scores(r.input_text), readability_scores(r.target_text)
    rows.append({"pair": i + 1,
                 "fk_technical": t["flesch_kincaid_grade"], "fk_plain": p["flesch_kincaid_grade"],
                 "smog_technical": t["smog_index"], "smog_plain": p["smog_index"],
                 "jargon_technical": t["jargon_density"], "jargon_plain": p["jargon_density"],
                 "words_technical": t["word_count"], "words_plain": p["word_count"]})
scores = pd.DataFrame(rows)
scores.round(2)

In [ ]:
summary = scores.drop(columns="pair").agg(["mean", "min", "max"]).round(2)
print("Plain rewrites at <= 8th grade:", (scores.fk_plain <= 8).sum(), "/ 20")
print("Pairs where the plain version scored HARDER:", (scores.fk_plain > scores.fk_technical).sum(), "/ 20")
summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = scores.pair
ax.bar(x - 0.2, scores.fk_technical, 0.4, label="Technical abstract", color="#14213D")
ax.bar(x + 0.2, scores.fk_plain, 0.4, label="Human plain rewrite", color="#1F7A6E")
ax.axhline(8, color="#C2410C", ls="--", label="Target: grade 8")
ax.set_xlabel("PLABA pair"); ax.set_ylabel("Flesch-Kincaid grade"); ax.set_xticks(x)
ax.set_title("Human plain-language rewrites are still far above grade 8")
ax.legend(); plt.tight_layout(); plt.show()

## Findings

- Plain rewrites drop FK by only ~1.7 grades on average (15.6 → 13.9); **0 of 20** reach grade 8.
- In some pairs the plain version scores *harder*: experts often keep the term and add an explanation in parentheses (e.g. "fibrosis" → "scarring (fibrosis)"), which makes sentences longer — and FK punishes length.
- So FK alone can **penalize good explaining**. PLABA is abstract-level research text, not discharge notes, so grade 8 may be unrealistic here.

**Implications for the Readability agent:** don't optimize FK blindly (it rewards short choppy sentences and deleting detail); report FK + SMOG + jargon together; check facts are kept (Verifier).

**Next:** run the same scoring on the single-LLM baseline outputs; test FK on short discharge-style text where grade 8 is realistic.